In [0]:
!pip install yfinance certifi lxml

#dbutils.library.restartPython()

## I. Import Libraries

In [0]:
from pyspark.sql.functions import lit, to_date, monotonically_increasing_id
from pyspark.sql.types import StructType, StructField, StringType, FloatType, IntegerType, LongType, DateType, TimestampType, DoubleType
import yfinance as yf
import pandas as pd
from datetime import datetime, timedelta
import time

## II. Retrieve the List of Stock Symbols

In [0]:
url = dbutils.secrets.get(scope="Capstone", key="sandp500url")
tables = pd.read_html(url)
sp500_table = tables[0]
sp500_symbols = sp500_table['Symbol'].tolist()
#display(sp500_table['Symbol'])

## III. Retrieve the Historical Prices Bronze Data and Store into the DBFS

In [0]:
jdbc_url = dbutils.secrets.get(scope="Capstone", key="DatabasejdbcUrl")#DatabasejdbcUrl
connection_properties = {
    "user": dbutils.secrets.get(scope="Capstone", key="DatabaseUsername"),
    "password": dbutils.secrets.get(scope="Capstone", key="DatabasePassword"),
    "driver": dbutils.secrets.get(scope="Capstone", key="DatabaseDriver")
}

old_data = spark.read.jdbc(
    url=jdbc_url,
    table="Bronze.Historical_Prices",
    properties=connection_properties
)

start_time = old_data.agg({'Date': 'max'}).collect()[0][0].date()
end_time = datetime.today().date()

In [0]:
list_of_symbol = sp500_table['Symbol'].to_list()

e_rdd = spark.sparkContext.emptyRDD()
# create empty dataframe
columns = StructType([
    StructField('Open', DoubleType(), True),
    StructField('High', DoubleType(), True),
    StructField('Low', DoubleType(), True),
    StructField('Close', DoubleType(), True),
    StructField('Volume', LongType(), True),
    StructField('Dividends', DoubleType(), True),
    StructField('Date', DateType(), True),
    StructField('Stock_Symbol', StringType(), True),
    StructField('Stock_Splits', DoubleType(), True),
])

df_historical_bronze = spark.createDataFrame(data=e_rdd, schema=columns)

for symbol in list_of_symbol:
    try:
        ticker = yf.Ticker(symbol)
        historical_data = ticker.history(start=start_time,end=end_time)
        historical_data_df = pd.DataFrame(historical_data)
        historical_data_df["Date"] = pd.to_datetime(historical_data_df.index, format='%Y-%m-%d')
        historical_data_df = spark.createDataFrame(historical_data_df)
        historical_data_df = historical_data_df.withColumn("Stock_Symbol", lit(symbol))
        historical_data_df = historical_data_df.withColumn("Stock_Splits", historical_data_df["Stock Splits"])
        historical_data_df = historical_data_df.drop("Stock Splits")
        df_historical_bronze = df_historical_bronze.union(historical_data_df)
        time.sleep(1)
    except Exception as e:
        print(f"Error: {e}")
        continue

In [0]:
#df_historical_bronze.to('/dbfs/FileStore/Bronze/Historical_Prices_Bronze.csv', index=False)
df_historical_bronze.write.mode("overwrite").parquet("/dbfs/FileStore/Bronze/Historical_Prices_Bronze.parquet")

In [0]:
df_spark_historical_prices = spark.read.parquet(
    "/dbfs/FileStore/Bronze/Historical_Prices_Bronze.parquet",
    header=True,
    inferSchema=True
)

In [0]:
df_spark_historical_prices.printSchema()

In [0]:
df_spark_historical_prices.write \
    .format("jdbc") \
    .option("url", jdbc_url) \
    .option("dbtable", "Bronze.Historical_Prices") \
    .option("user", connection_properties["user"]) \
    .option("password", connection_properties["password"]) \
    .option("driver", connection_properties["driver"]) \
    .mode("append") \
    .option("batchsize", 10000) \
    .option("numPartitions", 8) \
    .save()

print("Data successfully written to Azure SQL Database.")